# Deep Ensembles

In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv(r"C:\Users\Shivam\Desktop\Capstone_updated\Dataset\processed_train.csv")
np.random.seed(42)

engine_ids = df["Engine_ID"].unique()
np.random.shuffle(engine_ids)

train_ids = engine_ids[:70]
val_ids = engine_ids[70:85]
test_ids = engine_ids[85:]

train_df = df[df["Engine_ID"].isin(train_ids)].copy()
val_df = df[df["Engine_ID"].isin(val_ids)].copy()
test_df = df[df["Engine_ID"].isin(test_ids)].copy()

print("Training engines:", train_df["Engine_ID"].nunique())
print("Validation engines:", val_df["Engine_ID"].nunique())
print("Testing engines:", test_df["Engine_ID"].nunique())


Training engines: 70
Validation engines: 15
Testing engines: 15


In [3]:
FEATURES = [
    "Setting_1",
    "Setting_2",
    "Sensor_2",
    "Sensor_3",
    "Sensor_4",
    "Sensor_6",
    "Sensor_7",
    "Sensor_8",
    "Sensor_9",
    "Sensor_11",
    "Sensor_12",
    "Sensor_13",
    "Sensor_14",
    "Sensor_15",
    "Sensor_17",
    "Sensor_20",
    "Sensor_21"
]

TARGET = "RUL"


In [4]:
WINDOW = 30

def create_sequences(data, features, target, window=30):
    X = []
    y = []

    for engine_id, engine_data in data.groupby("Engine_ID"):

        engine_data = engine_data.sort_values("Cycle")

        feature_values = engine_data[features].values.astype("float32")
        target_values = engine_data[target].values.astype("float32")

        for i in range(window, len(engine_data)):
            X.append(feature_values[i-window:i])
            y.append(target_values[i])

    return np.array(X), np.array(y)


In [5]:
X_train, y_train = create_sequences(train_df, FEATURES, TARGET, WINDOW)
X_val, y_val = create_sequences(val_df, FEATURES, TARGET, WINDOW)
X_test, y_test = create_sequences(test_df, FEATURES, TARGET, WINDOW)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)


(12216, 30, 17)
(2720, 30, 17)
(2695, 30, 17)


## Train the ensemble

Same BiLSTM architecture as `baseline.ipynb`/`dropout.ipynb`, trained `N_MODELS` times with a different random seed each time. Different seeds land in different local minima, so the *disagreement* between the independently-trained models becomes the uncertainty signal -- no dropout-at-inference trick needed, each model in the ensemble is just a normal, fully-trained point predictor.

In [6]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Bidirectional,
    Dense,
    Dropout
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


def build_bilstm(seed):
    tf.random.set_seed(seed)

    model = Sequential([

        Input(shape=(WINDOW, len(FEATURES))),

        Bidirectional(
            LSTM(
                64,
                return_sequences=True
            )
        ),

        Dropout(0.2),

        Bidirectional(
            LSTM(32)
        ),

        Dropout(0.2),

        Dense(
            32,
            activation="relu"
        ),

        Dense(1)
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model


C:\Users\Shivam\AppData\Roaming\Python\Python310\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.2) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [7]:
N_MODELS = 5
SEEDS = [42, 43, 44, 45, 46]

ensemble_models = []

for seed in SEEDS:

    tf.keras.backend.clear_session()

    print(f"\n--- Training ensemble member, seed={seed} ---")

    model = build_bilstm(seed)

    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )

    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6
    )

    model.fit(
        X_train,
        y_train,

        validation_data=(
            X_val,
            y_val
        ),

        epochs=100,
        batch_size=64,

        shuffle=False,

        callbacks=[
            early_stopping,
            reduce_lr
        ],

        verbose=1
    )

    ensemble_models.append(model)

print(f"\nTrained {len(ensemble_models)} ensemble members.")




--- Training ensemble member, seed=42 ---
Epoch 1/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - loss: 3969.0669 - mae: 53.0159 - val_loss: 1762.8496 - val_mae: 37.5933 - learning_rate: 0.0010
Epoch 2/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - loss: 1779.4607 - mae: 37.4080 - val_loss: 1748.7897 - val_mae: 37.1141 - learning_rate: 0.0010
Epoch 3/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - loss: 1774.9136 - mae: 37.2679 - val_loss: 1748.7427 - val_mae: 37.1120 - learning_rate: 0.0010
Epoch 4/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - loss: 1773.3041 - mae: 37.2747 - val_loss: 1748.7092 - val_mae: 37.1051 - learning_rate: 0.0010
Epoch 5/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - loss: 1768.0868 - mae: 37.2481 - val_loss: 1748.4684 - val_mae: 37.0872 - learning_rate: 0.0010
Epoch 6/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - loss: 1572.6544 - mae: 34.5170 - val_loss: 1213.3851 - val_mae: 31.4969 - learning_rate: 0.0010
Epoch 7/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s

## Ensemble prediction

Same idea as `mc_dropout_predict()` in `dropout.ipynb`, but averaging across the `N_MODELS` independently-trained networks instead of across stochastic dropout passes of one network.

In [8]:
def ensemble_predict(models, X):

    predictions = np.array([
        model.predict(X, verbose=0).ravel()
        for model in models
    ])

    mean_pred = predictions.mean(axis=0)
    std_pred = predictions.std(axis=0)

    return mean_pred, std_pred


In [9]:
ens_mean, ens_std = ensemble_predict(
    ensemble_models,
    X_test
)

print("Predictions:", ens_mean.shape)
print("Uncertainty:", ens_std.shape)

print("\nMean uncertainty:", ens_std.mean())
print("Min uncertainty :", ens_std.min())
print("Max uncertainty :", ens_std.max())


Predictions: (2695,)
Uncertainty: (2695,)

Mean uncertainty: 5.189422
Min uncertainty : 0.23824102
Max uncertainty : 26.304434


In [10]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Held-out split:")
print("MAE :", mean_absolute_error(y_test, ens_mean))
print("RMSE:", np.sqrt(mean_squared_error(y_test, ens_mean)))
print("R\u00b2  :", r2_score(y_test, ens_mean))


Held-out split:
MAE : 10.72635269165039
RMSE: 13.966596262412205
R²  : 0.888350784778595


## Calibrate the accept/abstain threshold on `val_df`

**Important, and different from the current `dropout.ipynb`:** the threshold below is calibrated on `X_val` (engines the model never trained *or* was evaluated on for the numbers above), not on `X_test`. Calibrating a threshold on the same data you then report accuracy on is a soft form of leakage -- it makes the acceptance sweep look better than it would generalize. This matters even more once you add Conformal Prediction next, where a genuinely held-out calibration set isn't just best practice, it's a requirement for the method's coverage guarantee to mean anything. Worth going back and changing `dropout.ipynb`'s `mc_cal_std` (currently computed on `X_test`) to use `X_val` too, so all three methods are calibrated the same way for a fair three-way comparison.

In [11]:
ens_cal_mean, ens_cal_std = ensemble_predict(
    ensemble_models,
    X_val
)

ACCEPT_RATE = 0.80

ens_threshold = np.quantile(
    ens_cal_std,
    ACCEPT_RATE
)

print("Ensemble uncertainty threshold:", ens_threshold)


Ensemble uncertainty threshold: 7.486838


## Official test set (literature-comparable)

In [12]:
RUL_CAP = 125

test_df_final = pd.read_csv(r"C:\Users\Shivam\Desktop\Capstone_updated\Dataset\processed_test.csv")
output_df = pd.read_csv(r"C:\Users\Shivam\Desktop\Capstone_updated\Dataset\RUL_FD001.csv")
output_df["Engine_ID"] = np.arange(1, len(output_df) + 1)
output_df["RUL"] = output_df["RUL"].clip(upper=RUL_CAP)

def create_test_sequences(test_df, features, window=30):
    X, engine_ids = [], []
    for engine_id, engine_data in test_df.groupby("Engine_ID"):
        engine_data = engine_data.sort_values("Cycle")
        values = engine_data[features].values.astype("float32")
        if len(values) >= window:
            X.append(values[-window:])
            engine_ids.append(engine_id)
    return np.array(X), np.array(engine_ids)

X_test_final, test_engine_ids = create_test_sequences(test_df_final, FEATURES, WINDOW)


In [13]:
ens_test_mean, ens_test_std = ensemble_predict(
    ensemble_models,
    X_test_final
)

pred_df = pd.DataFrame({
    "Engine_ID": test_engine_ids,
    "Predicted_RUL": ens_test_mean,
    "Uncertainty": ens_test_std
})

comparison = pred_df.merge(output_df, on="Engine_ID")

actual = comparison["RUL"].values
ens_test_mean = comparison["Predicted_RUL"].values
ens_test_std = comparison["Uncertainty"].values


In [14]:
def nasa_score(actual, predicted):
    """NASA / PHM08 asymmetric RUL scoring function (Saxena et al., 2008).
    Penalizes late predictions (predicted RUL > actual RUL) more heavily
    than early ones, since a late prediction means maintenance would be
    scheduled after the engine could already have failed.
    """
    d = np.asarray(predicted) - np.asarray(actual)
    s = np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)
    return np.sum(s)


In [15]:
print("100% coverage:")
print("MAE  :", mean_absolute_error(actual, ens_test_mean))
print("RMSE :", np.sqrt(mean_squared_error(actual, ens_test_mean)))
print("R\u00b2   :", r2_score(actual, ens_test_mean))
print("Score:", nasa_score(actual, ens_test_mean))


100% coverage:
MAE  : 10.06637954711914
RMSE : 13.262048389680773
R²   : 0.8904754519462585
Score: 288.48282187185237


## Accept / abstain sweep

In [16]:
accept = ens_test_std <= ens_threshold

print("Accepted :", accept.sum())
print("Abstained:", (~accept).sum())
print("Coverage :", accept.mean())


Accepted : 83
Abstained: 17
Coverage : 0.83


In [17]:
y_true_accept = actual[accept]
y_pred_accept = ens_test_mean[accept]

mae = mean_absolute_error(y_true_accept, y_pred_accept)
rmse = np.sqrt(mean_squared_error(y_true_accept, y_pred_accept))
r2 = r2_score(y_true_accept, y_pred_accept)
score = nasa_score(y_true_accept, y_pred_accept)

print("Accepted predictions:")
print("MAE  :", mae)
print("RMSE :", rmse)
print("R\u00b2   :", r2)
print("Score:", score)


Accepted predictions:
MAE  : 9.53720474243164
RMSE : 12.889477931918805
R²   : 0.9041117429733276
Score: 218.55678694431595


In [18]:
for rate in [0.9, 0.8, 0.7]:

    threshold = np.quantile(
        ens_cal_std,
        rate
    )

    accept = ens_test_std <= threshold

    rmse = np.sqrt(
        mean_squared_error(
            actual[accept],
            ens_test_mean[accept]
        )
    )

    score = nasa_score(
        actual[accept],
        ens_test_mean[accept]
    )

    print(
        f"Acceptance={rate:.0%} | "
        f"Accepted={accept.sum()} | "
        f"RMSE={rmse:.3f} | "
        f"Score={score:.1f}"
    )


Acceptance=90% | Accepted=97 | RMSE=13.229 | Score=278.9
Acceptance=80% | Accepted=83 | RMSE=12.889 | Score=218.6
Acceptance=70% | Accepted=69 | RMSE=13.219 | Score=194.6


In [19]:
import os

METHOD = "deep_ensembles"
SCENARIO = "clean_capped"

RESULTS_DIR = r"C:\Users\Shivam\Desktop\Capstone_updated\results"
os.makedirs(RESULTS_DIR, exist_ok=True)

predicted = ens_test_mean
uncertainty = ens_test_std

results_df = pd.DataFrame({
    "Engine_ID": test_engine_ids,
    "Actual_RUL": actual,
    "Predicted_RUL": predicted,
    "Uncertainty": uncertainty
})

out_path = os.path.join(
    RESULTS_DIR,
    f"{METHOD}__{SCENARIO}.csv"
)

results_df.to_csv(out_path, index=False)

print(f"Saved {len(results_df)} rows -> {out_path}")

Saved 100 rows -> C:\Users\Shivam\Desktop\Capstone_updated\results\deep_ensembles__clean_capped.csv
